- Nama: Hilal Tsabitul Azmi Arbai
- NRP: 5054251008
- Dataset: Online Retail
- Algoritma: FP-Growth

In [90]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth

In [71]:
# df = pd.read_excel('Online Retail.xlsx')

In [72]:
# df.to_csv('online_retail.csv', index=False)
df= pd.read_csv('online_retail.csv', dtype={'InvoiceNo': str, 'StockCode': str})

In [73]:
df.shape

(541909, 8)

In [74]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [75]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [76]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [77]:
df['InvoiceNo']= df['InvoiceNo'].astype(str)
df['Description']= df['Description'].str.strip()

In [78]:
df= df.dropna(subset=['Description'])

In [79]:
batal = df[df['InvoiceNo'].astype(str).str.startswith('C')]

print(len(batal))
print(batal['Quantity'].describe())
batal.head()


9288
count     9288.000000
mean       -29.885228
std       1145.786965
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [80]:
df= df[~df['InvoiceNo'].str.startswith('C')]

In [81]:
df= df[(df['Quantity']>0) & (df['UnitPrice']>0)]

In [82]:
df.shape

(530104, 8)

In [83]:
df.nunique()

InvoiceNo      19960
StockCode       3922
Description     4015
Quantity         375
InvoiceDate    18499
UnitPrice       1291
CustomerID      4338
Country           38
dtype: int64

## Hasil Eksplorasi Data

Dataset Online Retail terdiri atas 541.909 baris dan 8 kolom, dengan 25.900 nomor invoice unik. Setiap baris mewakili satu produk dalam satu invoice, sehingga satu transaksi dapat terdiri atas beberapa baris.

Kolom yang digunakan untuk Association Rule adalah:

- **InvoiceNo**, sebagai identitas transaksi atau keranjang belanja.
- **Description**, sebagai item yang dibeli.

Kolom **Quantity** dan **UnitPrice** hanya digunakan saat pembersihan data, karena analisis ini hanya melihat ada atau tidaknya produk dalam transaksi. Nilai kosong pada **CustomerID** (135.080 baris atau 24,9%) tidak dihapus, karena pengelompokan transaksi dilakukan berdasarkan InvoiceNo.

Hasil pemeriksaan awal menunjukkan beberapa masalah data:

- **Description** kosong pada 1.454 baris.
- Terdapat 9.288 baris dengan InvoiceNo berawalan "C". Seluruh baris ini memiliki Quantity negatif, sesuai dengan statusnya sebagai transaksi yang dibatalkan.
- **Quantity** memiliki nilai minimum -80.995, dan **UnitPrice** memiliki nilai minimum -11.062,06. Nilai tersebut bukan penjualan normal.

## Hasil Pembersihan Data

| Tahap | Jumlah Baris | Jumlah Invoice |
|---|---:|---:|
| Data awal | 541.909 | 25.900 |
| Hapus Description kosong | 540.455 | 24.446 |
| Hapus transaksi dibatalkan | 531.167 | 20.610 |
| Hapus Quantity ≤ 0 | 530.693 | 20.136 |
| Hapus UnitPrice ≤ 0 | 530.104 | 19.960 |

Pembersihan menghapus 11.805 baris (2,2%) dan 5.940 invoice (22,9%). Penurunan invoice jauh lebih besar karena invoice yang dihapus umumnya hanya berisi sedikit baris. Contohnya, penghapusan 1.454 baris dengan Description kosong juga menghilangkan 1.454 invoice, yang berarti setiap invoice tersebut hanya memiliki satu baris. Sebaliknya, sebagian besar baris penjualan normal tetap dipertahankan.

Filter UnitPrice > 0 merupakan tahap tambahan di luar ketentuan tugas. Tahap ini menghapus 589 baris, sebagian di antaranya berdeskripsi seperti "check", "found", dan "adjustment" sehingga bukan penjualan yang sebenarnya.

Data hasil pembersihan masih memuat item non-produk, seperti POSTAGE, DOTCOM POSTAGE, Manual, CARRIAGE, dan Bank Charges. Item tersebut belum dihapus dan akan diperhatikan saat mengevaluasi aturan asosiasi.

In [84]:
transaksi= df.groupby('InvoiceNo')['Description'].apply(list)
transaksi.head()

InvoiceNo
536365    [WHITE HANGING HEART T-LIGHT HOLDER, WHITE MET...
536366    [HAND WARMER UNION JACK, HAND WARMER RED POLKA...
536367    [ASSORTED COLOUR BIRD ORNAMENT, POPPY'S PLAYHO...
536368    [JAM MAKING SET WITH JARS, RED COAT RACK PARIS...
536369                           [BATH BUILDING BLOCK WORD]
Name: Description, dtype: object

In [87]:
te= TransactionEncoder()
te_array= te.fit(transaksi).transform(transaksi)

basket= pd.DataFrame(te_array, columns=te.columns_, index=transaksi.index)
basket.shape

(19960, 4015)

In [88]:
print('persentse sel true', basket.values.mean()*100)

item_per_invoice= basket.sum(axis=1)
print(item_per_invoice.describe())

print(basket.sum().sort_values(ascending=False).head(10))

persentse sel true 0.6480070976323755
count    19960.000000
mean        26.017485
std         46.937574
min          1.000000
25%          6.000000
50%         15.000000
75%         29.000000
max       1108.000000
dtype: float64
WHITE HANGING HEART T-LIGHT HOLDER    2256
JUMBO BAG RED RETROSPOT               2089
REGENCY CAKESTAND 3 TIER              1988
PARTY BUNTING                         1685
LUNCH BAG RED RETROSPOT               1564
ASSORTED COLOUR BIRD ORNAMENT         1455
SET OF 3 CAKE TINS PANTRY DESIGN      1385
PACK OF 72 RETROSPOT CAKE CASES       1320
LUNCH BAG  BLACK SKULL.               1273
NATURAL SLATE HEART CHALKBOARD        1249
dtype: int64


In [93]:
hasil = []
for s in [0.05, 0.03, 0.02, 0.015, 0.01, 0.0075]:
    freq = fpgrowth(basket, min_support=s, use_colnames=True)
    aturan = association_rules(freq, metric='confidence', min_threshold=0.5)
    hasil.append({
        'min_support': s,
        'min_invoice': round(s * len(basket)),
        'itemset': len(freq),
        'aturan': len(aturan),
    })

pd.DataFrame(hasil)


,min_support,min_invoice,itemset,aturan
0,0.0500,998,33,0
1,0.0300,599,141,11
2,0.0200,399,379,61
3,0.0150,299,711,187
4,0.0100,200,1902,897
5,0.0075,150,4412,3781


In [94]:
frequent= fpgrowth(basket, min_support=0.01, use_colnames=True)

print('jumlah frequent itemset', len(frequent))
frequent['panjang']= frequent['itemsets'].apply(len)
print(frequent['panjang'].value_counts().sort_index())

jumlah frequent itemset 1902
panjang
1    809
2    841
3    233
4     18
5      1
Name: count, dtype: int64


In [ ]:
rules = association_rules(frequent, metric='confidence', min_threshold=0.5)

print('Jumlah aturan:', len(rules))
rules[['support', 'confidence', 'lift']].describe()

Jumlah aturan: 897


,support,confidence,lift
count,897.000000,897.000000,897.000000
mean,0.013418,0.653067,17.713849
std,0.004323,0.111356,14.333692
min,0.010020,0.500606,4.784125
25%,0.010671,0.558522,9.454304
50%,0.012024,0.630476,12.804926
75%,0.014279,0.734973,18.756475
max,0.041333,0.975728,79.518791


In [ ]:
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))

rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']].head()

,antecedents_str,consequents_str,support,confidence,lift
0,POPPY'S PLAYHOUSE BEDROOM,POPPY'S PLAYHOUSE KITCHEN,0.015681,0.739953,33.874900
1,POPPY'S PLAYHOUSE KITCHEN,POPPY'S PLAYHOUSE BEDROOM,0.015681,0.717890,33.874900
2,"JAM MAKING SET WITH JARS, SET OF 3 CAKE TINS P...",JAM MAKING SET PRINTED,0.010321,0.542105,9.311894
3,"JAM MAKING SET PRINTED, SET OF 3 CAKE TINS PAN...",JAM MAKING SET WITH JARS,0.010321,0.593660,10.467714
4,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED,0.032064,0.653061,12.402571


In [98]:
display(rules)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedents_str,consequents_str
0,(POPPY'S PLAYHOUSE BEDROOM),(POPPY'S PLAYHOUSE KITCHEN),0.021192,0.021844,0.015681,0.739953,33.874900,1.0,0.015218,3.761456,0.991492,0.573260,0.734145,0.728921,POPPY'S PLAYHOUSE BEDROOM,POPPY'S PLAYHOUSE KITCHEN
1,(POPPY'S PLAYHOUSE KITCHEN),(POPPY'S PLAYHOUSE BEDROOM),0.021844,0.021192,0.015681,0.717890,33.874900,1.0,0.015218,3.469594,0.992152,0.573260,0.711782,0.728921,POPPY'S PLAYHOUSE KITCHEN,POPPY'S PLAYHOUSE BEDROOM
2,"(JAM MAKING SET WITH JARS, SET OF 3 CAKE TINS ...",(JAM MAKING SET PRINTED),0.019038,0.058216,0.010321,0.542105,9.311894,1.0,0.009212,2.056769,0.909934,0.154192,0.513800,0.359693,"JAM MAKING SET WITH JARS, SET OF 3 CAKE TINS P...",JAM MAKING SET PRINTED
3,"(JAM MAKING SET PRINTED, SET OF 3 CAKE TINS PA...",(JAM MAKING SET WITH JARS),0.017385,0.056713,0.010321,0.593660,10.467714,1.0,0.009335,2.321422,0.920470,0.161822,0.569229,0.387819,"JAM MAKING SET PRINTED, SET OF 3 CAKE TINS PAN...",JAM MAKING SET WITH JARS
4,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE RED),0.049098,0.052655,0.032064,0.653061,12.402571,1.0,0.029479,2.730582,0.966842,0.460101,0.633778,0.631003,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
892,(SET OF 12 MINI LOAF BAKING CASES),(SET OF 6 SNACK LOAF BAKING CASES),0.022244,0.017735,0.011774,0.529279,29.842979,1.0,0.011379,2.086725,0.988480,0.417407,0.520780,0.596561,SET OF 12 MINI LOAF BAKING CASES,SET OF 6 SNACK LOAF BAKING CASES
893,(SET OF 6 SNACK LOAF BAKING CASES),(SET OF 6 TEA TIME BAKING CASES),0.017735,0.018938,0.010371,0.584746,30.877051,1.0,0.010035,2.362558,0.985084,0.394286,0.576730,0.566182,SET OF 6 SNACK LOAF BAKING CASES,SET OF 6 TEA TIME BAKING CASES
894,(SET OF 6 TEA TIME BAKING CASES),(SET OF 6 SNACK LOAF BAKING CASES),0.018938,0.017735,0.010371,0.547619,30.877051,1.0,0.010035,2.171322,0.986292,0.394286,0.539451,0.566182,SET OF 6 TEA TIME BAKING CASES,SET OF 6 SNACK LOAF BAKING CASES
895,(JUMBO BAG VINTAGE DOILY),(JUMBO BAG RED RETROSPOT),0.035621,0.104659,0.017836,0.500703,4.784125,1.0,0.014108,1.793203,0.820192,0.145663,0.442339,0.335560,JUMBO BAG VINTAGE DOILY,JUMBO BAG RED RETROSPOT


In [100]:
rules.describe()

,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
count,897.000000,897.000000,897.000000,897.000000,897.000000,897.0,897.000000,897.000000,897.000000,897.000000,897.000000,897.000000
mean,0.021261,0.051467,0.013418,0.653067,17.713849,1.0,0.012294,3.332733,0.939335,0.277452,0.634090,0.496039
std,0.008081,0.025720,0.004323,0.111356,14.333692,0.0,0.003802,2.512898,0.038512,0.154676,0.116754,0.122032
min,0.010321,0.010521,0.010020,0.500606,4.784125,1.0,0.008565,1.793203,0.814509,0.092414,0.442339,0.318529
25%,0.016032,0.035371,0.010671,0.558522,9.454304,1.0,0.009963,2.161197,0.916250,0.175546,0.537294,0.409826
50%,0.019589,0.051703,0.012024,0.630476,12.804926,1.0,0.011017,2.573322,0.944267,0.223988,0.611397,0.457769
75%,0.024198,0.059319,0.014279,0.734973,18.756475,1.0,0.012885,3.493508,0.968515,0.349309,0.713755,0.547792
max,0.063778,0.113026,0.041333,0.975728,79.518791,1.0,0.035719,39.742725,0.999400,0.863281,0.974838,0.926661


In [101]:
nonproduk = {'POSTAGE', 'DOTCOM POSTAGE', 'Manual', 'CARRIAGE', 'Bank Charges'}

ada_nonproduk = rules.apply(
    lambda r: bool((r['antecedents'] | r['consequents']) & nonproduk), axis=1
)
rules_produk = rules[~ada_nonproduk].copy()
print(len(rules_produk))

815


In [102]:
rules_produk['kombinasi'] = rules_produk.apply(
    lambda r: r['antecedents'] | r['consequents'], axis=1
)

rules_unik = (rules_produk
    .sort_values(['lift', 'confidence'], ascending=False)
    .drop_duplicates('kombinasi'))

print(len(rules_unik))
kolom = ['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'conviction']
rules_unik[kolom].head(10)

409


,antecedents_str,consequents_str,support,confidence,lift,conviction
379,"HERB MARKER PARSLEY, HERB MARKER ROSEMARY",HERB MARKER THYME,0.010170,0.944186,79.518791,17.703929
386,"HERB MARKER BASIL, HERB MARKER THYME",HERB MARKER ROSEMARY,0.010020,0.952381,79.206349,20.747495
372,HERB MARKER THYME,HERB MARKER ROSEMARY,0.011072,0.932489,77.552039,14.634394
374,HERB MARKER THYME,HERB MARKER PARSLEY,0.010671,0.898734,75.372833,9.757252
365,HERB MARKER PARSLEY,HERB MARKER ROSEMARY,0.010772,0.903361,75.129552,10.223403
368,HERB MARKER MINT,HERB MARKER PARSLEY,0.010621,0.883333,74.081232,8.469224
362,HERB MARKER ROSEMARY,HERB MARKER BASIL,0.010721,0.891667,73.544077,9.118853
370,HERB MARKER PARSLEY,HERB MARKER BASIL,0.010621,0.890756,73.468991,9.042863
363,HERB MARKER ROSEMARY,HERB MARKER MINT,0.010621,0.883333,73.463889,8.468365
376,HERB MARKER THYME,HERB MARKER BASIL,0.010521,0.886076,73.082958,8.671354


    ## Evaluasi dan Interpretasi Aturan Asosiasi

    Dengan min_support 0,01 dan min_confidence 0,5, diperoleh 1.902 frequent itemset dan 897 aturan. Sebanyak 82 aturan yang memuat item non-produk (POSTAGE, DOTCOM POSTAGE, Manual, CARRIAGE, dan Bank Charges) dikeluarkan dari pemilihan aturan terbaik karena bukan barang yang dapat dibundel, sehingga tersisa 815 aturan. Karena satu kombinasi produk dapat menghasilkan beberapa aturan dengan arah berbeda, hanya satu aturan terbaik per kombinasi yang dipertahankan, sehingga tersisa 409 aturan.

    ### Tiga Aturan Terbaik Berdasarkan Lift

    | No | Antecedent | Consequent | Invoice | Support | Confidence | Lift | Conviction |
    |---:|---|---|---:|---:|---:|---:|---:|
    | 1 | HERB MARKER PARSLEY, HERB MARKER ROSEMARY | HERB MARKER THYME | 203 | 0,0102 | 0,944 | 79,52 | 17,70 |
    | 2 | HERB MARKER BASIL, HERB MARKER THYME | HERB MARKER ROSEMARY | 200 | 0,0100 | 0,952 | 79,21 | 20,75 |
    | 3 | HERB MARKER THYME | HERB MARKER ROSEMARY | 221 | 0,0111 | 0,932 | 77,55 | 14,63 |

    **Aturan 1.** Sebanyak 94,4% invoice yang memuat HERB MARKER PARSLEY dan ROSEMARY juga memuat HERB MARKER THYME. Padahal, THYME secara umum hanya muncul di 1,19% invoice. Dengan lift 79,52, peluang THYME dibeli bersama kedua produk tersebut sekitar 79,5 kali lebih besar dibanding jika pembeliannya independen.

    **Aturan 2.** Sebanyak 95,2% invoice yang memuat HERB MARKER BASIL dan THYME juga memuat HERB MARKER ROSEMARY, sedangkan ROSEMARY secara umum hanya muncul di 1,20% invoice. Aturan ini memiliki confidence dan conviction tertinggi di antara ketiga aturan, sehingga implikasinya paling kuat.

    **Aturan 3.** Sebanyak 93,2% invoice yang memuat HERB MARKER THYME juga memuat HERB MARKER ROSEMARY. Aturan ini paling sederhana karena hanya memerlukan satu produk sebagai pemicu, dan didukung oleh invoice terbanyak di antara ketiganya, yaitu 221 invoice.

    ### Makna dalam Konteks Bisnis

    Ketiga aturan berasal dari keluarga produk yang sama. Terdapat enam varian HERB MARKER, masing-masing hanya muncul di 209 sampai 242 invoice, dan keduabelas aturan unik teratas seluruhnya berasal dari keluarga ini. Hal ini menunjukkan bahwa pelanggan diduga membeli varian-varian tersebut sebagai satu set, meskipun data transaksi tidak dapat memastikan alasan pembeliannya.

    Sebagai toko online, tindakan yang sesuai adalah menawarkan paket satu set HERB MARKER dan menampilkan varian lain sebagai rekomendasi di halaman produk atau keranjang. Karena lebih dari 90% pembeli sudah mengambil produk pasangannya, manfaat utamanya adalah mempermudah pembelian dan menjangkau sisa pembeli yang belum mengambil seluruh varian, bukan menciptakan permintaan baru dalam jumlah besar.

    ### Temuan Tambahan

    Pola yang sama juga muncul pada keluarga produk lain. Aturan REGENCY TEA PLATE PINK → REGENCY TEA PLATE GREEN didukung 278 invoice, dengan confidence 0,911 dan lift 47,75. Hal ini menunjukkan pelanggan cenderung membeli produk serupa dalam warna berbeda, sehingga rekomendasi varian warna dapat diterapkan lebih luas.

    Sebaliknya, aturan dengan support terbesar di antara aturan satu produk ke satu produk adalah JUMBO BAG PINK POLKADOT → JUMBO BAG RED RETROSPOT, yang didukung 825 invoice, dengan confidence 0,677 dan lift 6,47. Lift-nya jauh lebih rendah daripada HERB MARKER, tetapi jangkauannya sekitar empat kali lebih luas. Dari sisi dampak bisnis, aturan ini dapat lebih bernilai meskipun peringkat lift-nya lebih rendah.

    ### Kesimpulan Evaluasi

    Lift tinggi tidak selalu berarti aturan paling berdampak, karena lift cenderung tinggi pada produk yang jarang dibeli. Selain itu, filter confidence 0,5 membuat seluruh 897 aturan memiliki lift minimal 4,78, sehingga lift tinggi pada dataset ini perlu dibaca bersama support dan confidence. HERB MARKER menunjukkan keterkaitan terkuat, sedangkan JUMBO BAG menawarkan jangkauan yang lebih luas.
